# Model Development

مقارنة مختصرة وواضحة بين نماذج شجرية مناسبة للمشروع:
- `logistic_regression_original` على البيانات الأصلية مع `class_weight='balanced'`
- `logistic_regression_smote` على بيانات تدريب متوازنة باستخدام `SMOTE`
- `decision_tree_original` على البيانات الأصلية مع `class_weight='balanced'`
- `decision_tree_smote` على بيانات تدريب متوازنة باستخدام `SMOTE`
- `random_forest_original` على البيانات الأصلية مع `class_weight='balanced'`
- `random_forest_smote` على بيانات تدريب متوازنة باستخدام `SMOTE`

التركيز هنا على وضوح المقارنة بين `Decision Tree` و`Random Forest` مع الاهتمام بمقاييس الفئة الإيجابية مثل `Recall` و `F1-score`.

In [1]:
import json
import pickle
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

In [2]:
BASE_DIR = Path.cwd()
RAW_DATA_PATH = BASE_DIR / "data" / "student_lifestyle_100k.csv"

with open("preprocessed_original.pkl", "rb") as f:
    X_train_original, X_val_scaled, X_test_scaled, y_train_original, y_val, y_test = pickle.load(f)

with open("preprocessed_smote.pkl", "rb") as f:
    X_train_smote, _, _, y_train_smote, _, _ = pickle.load(f)

feature_names = (
    pd.read_csv(RAW_DATA_PATH, nrows=1)
    .drop(columns=["Student_ID", "Depression"])
    .columns
    .tolist()
)

feature_names


['Age',
 'Gender',
 'Department',
 'CGPA',
 'Sleep_Duration',
 'Study_Hours',
 'Social_Media_Hours',
 'Physical_Activity',
 'Stress_Level']

In [3]:
train_distribution_df = pd.concat(
    {
        "original_count": pd.Series(y_train_original).value_counts().sort_index(),
        "original_ratio": pd.Series(y_train_original).value_counts(normalize=True).sort_index(),
        "smote_count": pd.Series(y_train_smote).value_counts().sort_index(),
        "smote_ratio": pd.Series(y_train_smote).value_counts(normalize=True).sort_index(),
    },
    axis=1,
)

train_distribution_df.index.name = "class"
train_distribution_df.round(4)


,original_count,original_ratio,smote_count,smote_ratio
class,,,,
0,71950,0.8994,71950,0.5
1,8050,0.1006,71950,0.5


In [4]:
def evaluate_predictions(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_class_0": report["0"]["precision"],
        "recall_class_0": report["0"]["recall"],
        "f1_class_0": report["0"]["f1-score"],
        "precision_class_1": report["1"]["precision"],
        "recall_class_1": report["1"]["recall"],
        "f1_class_1": report["1"]["f1-score"],
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "report": report,
        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]],
    }


def run_experiment(name, model, X_train_used, y_train_used):
    model.fit(X_train_used, y_train_used)

    y_val_pred = model.predict(X_val_scaled)
    y_test_pred = model.predict(X_test_scaled)

    val_metrics = evaluate_predictions(y_val, y_val_pred)
    test_metrics = evaluate_predictions(y_test, y_test_pred)

    return {
        "experiment": name,
        "model_type": type(model).__name__,
        "train_samples": int(len(y_train_used)),
        "train_positive_ratio": float(pd.Series(y_train_used).mean()),
        "val_accuracy": val_metrics["accuracy"],
        "val_precision_class_1": val_metrics["precision_class_1"],
        "val_recall_class_1": val_metrics["recall_class_1"],
        "val_f1_class_1": val_metrics["f1_class_1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_precision_class_0": test_metrics["precision_class_0"],
        "test_recall_class_0": test_metrics["recall_class_0"],
        "test_f1_class_0": test_metrics["f1_class_0"],
        "test_precision_class_1": test_metrics["precision_class_1"],
        "test_recall_class_1": test_metrics["recall_class_1"],
        "test_f1_class_1": test_metrics["f1_class_1"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],
        "val_report": val_metrics["report"],
        "test_report": test_metrics["report"],
        "test_confusion_matrix": test_metrics["confusion_matrix"],
        "model": model,
    }


experiments = {
    "decision_tree_original": (
        DecisionTreeClassifier(
            random_state=42,
            max_depth=8,
            min_samples_leaf=20,
                    ),
        X_train_original,
        y_train_original,
    ),
    "decision_tree_smote": (
        DecisionTreeClassifier(
            random_state=42,
            max_depth=8,
            min_samples_leaf=20,
        ),
        X_train_smote,
        y_train_smote,
    ),
    "random_forest_original": (
        RandomForestClassifier(
            n_estimators=250,
            random_state=42,
            max_depth=12,
            min_samples_leaf=5,
                        n_jobs=-1,
        ),
        X_train_original,
        y_train_original,
    ),
    "random_forest_smote": (
        RandomForestClassifier(
            n_estimators=250,
            random_state=42,
            max_depth=12,
            min_samples_leaf=5,
            n_jobs=-1,
        ),
        X_train_smote,
        y_train_smote,
    ),
}

trained_models = {    "logistic_regression_original": (
        LogisticRegression(
            random_state=42,
                        max_iter=1000,
            n_jobs=-1,
        ),
        X_train_original,
        y_train_original,
    ),
    "logistic_regression_smote": (
        LogisticRegression(
            random_state=42,
            max_iter=1000,
            n_jobs=-1,
        ),
        X_train_smote,
        y_train_smote,
    ),
}
results = []

for name, (model, X_train_used, y_train_used) in experiments.items():
    result = run_experiment(name, model, X_train_used, y_train_used)
    trained_models[name] = result.pop("model")
    results.append(result)

results_df = (
    pd.DataFrame(results)
    .sort_values(by=["test_f1_class_1", "test_recall_class_1"], ascending=False)
    .reset_index(drop=True)
)

results_df[["experiment", "model_type", "test_accuracy", "test_recall_class_1", "test_f1_class_1"]].round(4)


,experiment,model_type,test_accuracy,test_recall_class_1,test_f1_class_1
0,decision_tree_smote,DecisionTreeClassifier,0.7486,0.5388,0.3013
1,random_forest_smote,RandomForestClassifier,0.7576,0.5040,0.2949
2,decision_tree_original,DecisionTreeClassifier,0.8983,0.0308,0.0575
3,random_forest_original,RandomForestClassifier,0.8991,0.0249,0.0472


## Summary Comparison

هذا الجدول هو المرجع السريع لاتخاذ القرار، مع التركيز على `test_f1_class_1` و `test_recall_class_1`.

In [5]:
summary_columns = [
    "experiment",
    "model_type",
    "train_positive_ratio",
    "test_accuracy",
    "test_precision_class_1",
    "test_recall_class_1",
    "test_f1_class_1",
    "test_macro_f1",
    "test_weighted_f1",
]

summary_df = (
    results_df[summary_columns]
    .sort_values(by=["test_f1_class_1", "test_recall_class_1"], ascending=False)
    .reset_index(drop=True)
    .round(4)
)

summary_df


,experiment,model_type,train_positive_ratio,test_accuracy,test_precision_class_1,test_recall_class_1,test_f1_class_1,test_macro_f1,test_weighted_f1
0,decision_tree_smote,DecisionTreeClassifier,0.5000,0.7486,0.2091,0.5388,0.3013,0.5740,0.7919
1,random_forest_smote,RandomForestClassifier,0.5000,0.7576,0.2085,0.5040,0.2949,0.5743,0.7974
2,decision_tree_original,DecisionTreeClassifier,0.1006,0.8983,0.4247,0.0308,0.0575,0.5019,0.8568
3,random_forest_original,RandomForestClassifier,0.1006,0.8991,0.4717,0.0249,0.0472,0.4970,0.8562


## Detailed Comparison

هذا الجدول يبقي التفاصيل المهمة كلها في مكان واحد بدل تكرار تقارير طويلة لكل تجربة.

In [6]:
detailed_columns = [
    "experiment",
    "model_type",
    "train_samples",
    "train_positive_ratio",
    "val_accuracy",
    "val_precision_class_1",
    "val_recall_class_1",
    "val_f1_class_1",
    "test_accuracy",
    "test_precision_class_0",
    "test_recall_class_0",
    "test_f1_class_0",
    "test_precision_class_1",
    "test_recall_class_1",
    "test_f1_class_1",
    "test_macro_f1",
    "test_weighted_f1",
    "tn",
    "fp",
    "fn",
    "tp",
]

detailed_df = (
    results_df[detailed_columns]
    .sort_values(by=["test_f1_class_1", "test_recall_class_1"], ascending=False)
    .reset_index(drop=True)
    .round(4)
)

detailed_df


,experiment,model_type,train_samples,train_positive_ratio,val_accuracy,val_precision_class_1,val_recall_class_1,val_f1_class_1,test_accuracy,test_precision_class_0,...,test_f1_class_0,test_precision_class_1,test_recall_class_1,test_f1_class_1,test_macro_f1,test_weighted_f1,tn,fp,fn,tp
0,decision_tree_smote,DecisionTreeClassifier,143900,0.5000,0.7530,0.2215,0.5785,0.3203,0.7486,0.9374,...,0.8467,0.2091,0.5388,0.3013,0.5740,0.7919,6944,2050,464,542
1,random_forest_smote,RandomForestClassifier,143900,0.5000,0.7625,0.2243,0.5537,0.3193,0.7576,0.9341,...,0.8536,0.2085,0.5040,0.2949,0.5743,0.7974,7069,1925,499,507
2,decision_tree_original,DecisionTreeClassifier,80000,0.1006,0.9002,0.5741,0.0308,0.0585,0.8983,0.9018,...,0.9463,0.4247,0.0308,0.0575,0.5019,0.8568,8952,42,975,31
3,random_forest_original,RandomForestClassifier,80000,0.1006,0.9001,0.6207,0.0179,0.0348,0.8991,0.9014,...,0.9467,0.4717,0.0249,0.0472,0.4970,0.8562,8966,28,981,25


## Focused Best-Model Report

بدل عرض تقرير كامل لكل تجربة، نعرض هنا التقرير التفصيلي للنموذج الأفضل حسب `test_f1_class_1`.

In [7]:
focus_model_name = summary_df.iloc[0]["experiment"]
focus_row = results_df.loc[results_df["experiment"] == focus_model_name].iloc[0]

focus_test_report_df = (
    pd.DataFrame(focus_row["test_report"])
    .T
    .loc[["0", "1", "accuracy", "macro avg", "weighted avg"]]
    .round(4)
)

focus_confusion_df = pd.DataFrame(
    focus_row["test_confusion_matrix"],
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
)

display(pd.DataFrame([focus_row[["experiment", "model_type", "test_accuracy", "test_recall_class_1", "test_f1_class_1"]]]).round(4))
display(focus_test_report_df)
focus_confusion_df


,experiment,model_type,test_accuracy,test_recall_class_1,test_f1_class_1
0,decision_tree_smote,DecisionTreeClassifier,0.7486,0.5388,0.3013


,precision,recall,f1-score,support
0,0.9374,0.7721,0.8467,8994.0000
1,0.2091,0.5388,0.3013,1006.0000
accuracy,0.7486,0.7486,0.7486,0.7486
macro avg,0.5732,0.6554,0.5740,10000.0000
weighted avg,0.8641,0.7486,0.7919,10000.0000


,Predicted 0,Predicted 1
Actual 0,6944,2050
Actual 1,464,542


## Feature Importance

هذه الخلية تساعد في تفسير النتائج، وهي نقطة مهمة في التقرير والعرض النهائي.

In [8]:
best_model_name = summary_df.iloc[0]["experiment"]
best_model = trained_models[best_model_name]

feature_importance_df = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "importance": best_model.feature_importances_ if hasattr(best_model, 'feature_importances_') else abs(best_model.coef_[0]),
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance_df


,feature,importance
0,CGPA,0.405425
1,Department,0.300099
2,Stress_Level,0.141801
3,Age,0.138802
4,Physical_Activity,0.006861
5,Sleep_Duration,0.004489
6,Gender,0.000953
7,Study_Hours,0.000838
8,Social_Media_Hours,0.000731


## Saved Models

يتم حفظ النموذج الأفضل إجمالًا، مع حفظ أفضل `Decision Tree` وأفضل `Random Forest` بشكل منفصل، بالإضافة إلى ملفات CSV وJSON المساعدة.

In [9]:
logistic_regression_best_name = (
    results_df[results_df["model_type"] == "LogisticRegression"].iloc[0]["experiment"]
    if not results_df[results_df["model_type"] == "LogisticRegression"].empty else None
)
decision_tree_best_name = (
    results_df[results_df["model_type"] == "DecisionTreeClassifier"].iloc[0]["experiment"]
)
random_forest_best_name = (
    results_df[results_df["model_type"] == "RandomForestClassifier"].iloc[0]["experiment"]
)

with open("saved_tree_best_model.pkl", "wb") as f:
    pickle.dump(trained_models[best_model_name], f)

if logistic_regression_best_name:
    with open("saved_logistic_regression.pkl", "wb") as f:
        pickle.dump(trained_models[logistic_regression_best_name], f)

with open("saved_decision_tree.pkl", "wb") as f:
    pickle.dump(trained_models[decision_tree_best_name], f)

with open("saved_random_forest.pkl", "wb") as f:
    pickle.dump(trained_models[random_forest_best_name], f)

summary_df.to_csv("tree_model_comparison.csv", index=False)
feature_importance_df.to_csv("best_tree_feature_importance.csv", index=False)

details_payload = {
    "best_experiment": best_model_name,
    "decision_tree_best_experiment": decision_tree_best_name,
    "random_forest_best_experiment": random_forest_best_name,
    "logistic_regression_best_experiment": logistic_regression_best_name,
    "results": results,
}

with open("tree_model_details.json", "w", encoding="utf-8") as f:
    json.dump(details_payload, f, ensure_ascii=False, indent=2)

{
    "best_experiment": best_model_name,
    "decision_tree_best_experiment": decision_tree_best_name,
    "random_forest_best_experiment": random_forest_best_name,
    "logistic_regression_best_experiment": logistic_regression_best_name,
    "saved_files": [
        "saved_tree_best_model.pkl",
        "saved_decision_tree.pkl",
        "saved_random_forest.pkl",
        "saved_logistic_regression.pkl",
        "tree_model_comparison.csv",
        "best_tree_feature_importance.csv",
        "tree_model_details.json",
    ],
}


{'best_experiment': 'decision_tree_smote',
 'decision_tree_best_experiment': 'decision_tree_smote',
 'random_forest_best_experiment': 'random_forest_smote',
 'logistic_regression_best_experiment': None,
 'saved_files': ['saved_tree_best_model.pkl',
  'saved_decision_tree.pkl',
  'saved_random_forest.pkl',
  'saved_logistic_regression.pkl',
  'tree_model_comparison.csv',
  'best_tree_feature_importance.csv',
  'tree_model_details.json']}